<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 10 · Object-Oriented Trading System Design

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook demonstrates how the Chapter 10 OLS strategy is wrapped in object-oriented components.

- Use `EODDataHandler` to obtain a clean price series.
- Fit `OLSStrategy` and generate positions.
- Run `PortfolioBacktester` and compare strategy and benchmark.


In [ ]:
import sys
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch10_oop_ols_backtest as ch10  # object-oriented OLS backtest


### 1. Data Handler and Strategy

We construct a data handler and an OLS strategy instance configured for EURUSD, then fit the model and inspect the test window.

In [ ]:
data_handler = ch10.EODDataHandler(symbol="EURUSD")
strategy = ch10.OLSStrategy(
    symbol="EURUSD",
    n_lags=5,
    entry_threshold=0.002,
    leverage=1.0,
    train_fraction=0.7,
)

_, _, _ = strategy.fit()  # fit coefficients on the training window
pos_ols = strategy.generate_positions()
pos_ols.head()


### 2. Portfolio Backtest

The backtester reuses the Chapter 7 cost-aware engine and reports both benchmark and OLS-based equity curves.

In [ ]:
backtester = ch10.PortfolioBacktester(
    data_handler=data_handler,
    strategy=strategy,
)

net_bh, net_ols = backtester.run()
wealth_bh = (1.0 + net_bh).cumprod()
wealth_ols = (1.0 + net_ols).cumprod()

fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot(wealth_bh.index, wealth_bh.values, label="buy-and-hold")
ax.plot(wealth_ols.index, wealth_ols.values, label="OLS strategy")
ax.set_ylabel("wealth (normalised)")
ax.set_xlabel("date")
ax.set_title("EURUSD: object-oriented OLS vs buy-and-hold")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- Separating data handling, strategy logic, and portfolio evaluation clarifies responsibilities.
- The object-oriented layer remains thin: it mostly wires existing helpers together cleanly.
- This structure is a natural stepping stone towards the fully event-based engine in Chapter 11.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">